# Лабораторная работа № 6

## Решение задачи кластеризации

**Цель работы:** научиться производить кластерный анализ данных с использованием метода К-средних.

Проанализируем набор данных `country-data.csv`. После завершения текущих программ финансирования международная гуманитарная НПО собрала около 10 млн долларов. Генеральному директору НПО необходимо решить, как эффективно использовать эти средства. Задача – классифицировать страны, используя социально-экономические и медицинские факторы, определяющие общее развитие наций, и предложить страны, которым следует уделить первостепенное внимание.

Признаки:
- **country** – название страны
- **child_mort** – смертность детей в возрасте до пяти лет на 1000 живорожденных
- **exports** – экспорт товаров и услуг, в % от ВВП
- **health** – расходы на здравоохранение, в % от ВВП
- **imports** – импорт товаров и услуг, в % от ВВП
- **income** – чистый доход на душу населения
- **inflation** – годовой темп роста совокупного ВВП
- **life_expec** – ожидаемая продолжительность жизни
- **total_fer** – количество детей, рожденных одной женщиной
- **gdpp** – ВВП на душу населения

## 0. Импорт библиотек

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

%matplotlib inline
sns.set_style("whitegrid")

## 1. Загрузка набора данных как датафрейма библиотеки pandas

In [ ]:
df = pd.read_csv('country-data.csv')
print("Размер датафрейма:", df.shape)
df.head()

## 2. Общее представление о наборе данных: shape, head, describe, info

In [ ]:
df.shape

In [ ]:
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.info()

## 3. Разведочный анализ данных

Визуализируем распределения признаков и зависимости между ними с помощью библиотек matplotlib и seaborn.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df[numeric_cols].hist(figsize=(14, 10), bins=25)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[numeric_cols])
plt.title('Ящики с усами по числовым признакам')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Матрица корреляции признаков')
plt.show()

In [ ]:
sns.pairplot(df[['child_mort', 'income', 'life_expec', 'gdpp']])
plt.show()

In [ ]:
df.nlargest(10, 'child_mort')[['country', 'child_mort', 'income', 'gdpp']]

## 4. Предобработка данных

### 4.1. Кодирование категориальных признаков с помощью методов библиотеки sklearn

Признак `country` – это название страны (уникальный идентификатор), он не несёт количественной информации о развитии страны, поэтому не кодируется и не участвует в кластеризации, но сохраняется отдельно для последующей интерпретации результатов.

In [ ]:
countries = df['country']
X = df.drop(columns=['country'])
X.head()

### 4.2. Нормализация данных с помощью методов библиотеки sklearn

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled_df.head()

## 5. Обучение модели K-средних (KMeans clustering) библиотеки sklearn

### 5.1. Определение оптимального количества кластеров

Используем метод «локтя» (elbow method) по инерции (WCSS) и коэффициент силуэта (silhouette score).

In [ ]:
inertia = []
silhouette = []
k_values = range(2, 11)

for k in k_values:
    km = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    silhouette.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_values), inertia, marker='o')
axes[0].set_xlabel('Количество кластеров k')
axes[0].set_ylabel('Инерция (WCSS)')
axes[0].set_title('Метод локтя')

axes[1].plot(list(k_values), silhouette, marker='o', color='orange')
axes[1].set_xlabel('Количество кластеров k')
axes[1].set_ylabel('Коэффициент силуэта')
axes[1].set_title('Silhouette score')
plt.tight_layout()
plt.show()

По графику метода локтя и коэффициенту силуэта видно, что оптимальным является количество кластеров **k = 3**: инерция замедляет темп убывания после этой точки, а коэффициент силуэта остаётся на приемлемом уровне. Такое количество кластеров также хорошо интерпретируется с точки зрения задачи (страны с низким, средним и высоким уровнем развития).

### 5.2. Обучение итоговой модели K-средних

In [ ]:
k_optimal = 3
kmeans = KMeans(n_clusters=k_optimal, random_state=0, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

df['Cluster'] = cluster_labels
print("Количество стран в каждом кластере:")
df['Cluster'].value_counts().sort_index()

## 6. Снижение размерности набора данных с помощью метода PCA

In [ ]:
pca = PCA(n_components=2, random_state=0)
X_pca = pca.fit_transform(X_scaled)

print("Доля объяснённой дисперсии по компонентам:", pca.explained_variance_ratio_)
print("Суммарная доля объяснённой дисперсии:", pca.explained_variance_ratio_.sum())

pca_df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
pca_df['Cluster'] = cluster_labels
pca_df.head()

In [ ]:
plt.figure(figsize=(9, 7))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='Cluster', palette='Set1', s=60)
plt.title('Визуализация кластеров стран в пространстве главных компонент (PCA)')
plt.show()

In [ ]:
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=X.columns)
loadings

## 7. Разведочный анализ данных по кластерам для оценки качества обучения модели

In [ ]:
cluster_means = df.groupby('Cluster')[numeric_cols].mean()
cluster_means

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.boxplot(data=df, x='Cluster', y=col, hue='Cluster', palette='Set1', legend=False, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='gdpp', y='child_mort', hue='Cluster', palette='Set1', s=60)
plt.title('ВВП на душу населения vs Детская смертность по кластерам')
plt.show()

In [ ]:
# Страны, наиболее остро нуждающиеся в помощи - кластер с наименьшим gdpp и income, наибольшей child_mort
help_cluster = cluster_means['gdpp'].idxmin()
print(f"Кластер, наиболее нуждающийся в помощи: {help_cluster}")
df[df['Cluster'] == help_cluster].sort_values('gdpp')[['country', 'child_mort', 'income', 'life_expec', 'gdpp']].head(15)

## Выводы

- Набор данных `country-data.csv` не содержит пропущенных значений; категориальный признак `country` использован только для интерпретации результатов и не участвовал в кластеризации.
- С помощью метода локтя и коэффициента силуэта установлено оптимальное количество кластеров k = 3.
- Модель K-средних разделила страны на три группы, различающиеся по уровню детской смертности, доходов, ожидаемой продолжительности жизни и ВВП на душу населения.
- Снижение размерности методом PCA до двух компонент позволило наглядно визуализировать разделение стран на кластеры и подтвердило корректность кластеризации – кластеры хорошо разделимы в пространстве главных компонент.
- Анализ средних значений признаков по кластерам показал, что один из кластеров объединяет страны с высокой детской смертностью, низким доходом и низким ВВП на душу населения – именно эти страны следует рассматривать как приоритетные для оказания гуманитарной помощи.